# Fase 3 · Parte 3: Modelo comparativo
**Dataset de propiedades en venta — Mérida, Yucatán**

## Propósito
Comparar Random Forest y Gradient Boosting contra el baseline de regresión
lineal (Parte 2) siguiendo una estrategia de dos etapas: primero con el
mismo feature set del baseline para aislar el efecto del modelo, luego
incorporando `ratio_construccion_terreno` para medir el aporte del feature
nuevo de forma independiente. Se usa CV anidada para tunear hiperparámetros
sin inflar las métricas de evaluación.

## Entrada y salida
- **Entrada:** `data/propiedades_modelo.csv` — 59 propiedades con features
  construidas en la Parte 1.
- **Salida:** tabla comparativa de los cuatro modelos (OLS baseline, RF base,
  GB base, mejor modelo con ratio) con RMSE y MAE en escala original (MXN).

## Referencia del baseline (Parte 2)
- RMSE: 6.90M ± 2.05M MXN
- MAE:  3.19M ± 0.84M MXN
- Protocolo: StratifiedKFold(k=5), estratificado por colonia

## Contenido
1. Configuración
2. Carga de datos
3. Preparación de feature sets
   - 3.1 Feature set base (igual que baseline OLS)
   - 3.2 Feature set extendido (con ratio_construccion_terreno)
   - 3.3 Codificación de colonia
4. Etapa 1 — Modelos con feature set base
   - 4.1 Random Forest
   - 4.2 Gradient Boosting
5. Etapa 2 — Modelos con feature set extendido
   - 5.1 Random Forest con ratio
   - 5.2 Gradient Boosting con ratio
6. Tabla comparativa
7. Conclusiones

## 1. Configuración
Importamos las librerías y definimos la ruta a los datos. Se agregan
los módulos de scikit-learn para modelos de árbol, búsqueda de
hiperparámetros y validación cruzada anidada.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import (
    StratifiedKFold, RepeatedStratifiedKFold, GridSearchCV
)
from sklearn.metrics import mean_squared_error, mean_absolute_error

pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "propiedades_modelo.csv"

print("Raíz del proyecto:", PROJECT_ROOT)
print("¿Existe el CSV de modelado?:", DATA_PATH.exists())

Raíz del proyecto: /Users/edson/Documents/GitHub/dataset-inmuebles-merida
¿Existe el CSV de modelado?: True


## 2. Carga de datos
Se carga el conjunto de modelado generado en la Fase 3 · Parte 1
(`02_feature_engineering.ipynb`). Verificación de forma y columnas
esperadas.

In [2]:
df = pd.read_csv(DATA_PATH)

print("Forma:", df.shape)  # esperado: (59, 21)
print("\nFaltantes en columnas de modelado:")
cols_modelo = [
    "log_precio", "m2_construccion", "m2_terreno", "recamaras",
    "banos", "estacionamientos", "es_preventa", "es_casa",
    "tiene_piscina", "tiene_cuarto_servicio",
    "es_una_planta", "tiene_mantenimiento_con_monto", "colonia",
]
print(df[cols_modelo].isna().sum())

Forma: (59, 21)

Faltantes en columnas de modelado:
log_precio                       0
m2_construccion                  0
m2_terreno                       0
recamaras                        0
banos                            0
estacionamientos                 0
es_preventa                      0
es_casa                          0
tiene_piscina                    0
tiene_cuarto_servicio            0
es_una_planta                    0
tiene_mantenimiento_con_monto    0
colonia                          0
dtype: int64


## 3. Preparación de feature sets

### 3.1 Feature set base (igual que baseline OLS)
Se mantiene `log_m2_construccion` del baseline para que la comparación
aísle el efecto del modelo y no el del feature. Aunque RF y GB son
robustos al punto de leverage que motivó la transformación, usamos la
misma forma para no mezclar dos cambios a la vez.

In [3]:
FEATURES_BASE = [
    "log_m2_construccion",
    "recamaras",
    "banos",
    "estacionamientos",
    "es_preventa",
    "es_casa",
    "tiene_piscina",
    "tiene_cuarto_servicio",
    "es_una_planta",
    "tiene_mantenimiento_con_monto",
]

df["log_m2_construccion"] = np.log(df["m2_construccion"])

TARGET = "log_precio"
STRAT_COL = "colonia"

print("Features base:", FEATURES_BASE)
print(f"\nTotal: {len(FEATURES_BASE)} (+ dummies de colonia en 3.3)")

Features base: ['log_m2_construccion', 'recamaras', 'banos', 'estacionamientos', 'es_preventa', 'es_casa', 'tiene_piscina', 'tiene_cuarto_servicio', 'es_una_planta', 'tiene_mantenimiento_con_monto']

Total: 10 (+ dummies de colonia en 3.3)


### 3.2 Feature set extendido (con ratio_construccion_terreno)
Se agrega `ratio_construccion_terreno` = m2_construccion / m2_terreno como
feature adicional. Captura la densidad de edificación: qué proporción del
terreno está construida. Esta variable fue descartada en la Parte 1 por
multicolinealidad con m2_terreno en OLS; RF y GB son robustos a ese problema
y pueden aprovechar la información adicional. Se mantiene `log_m2_construccion`
para preservar la información de tamaño absoluto.

In [4]:
df["ratio_construccion_terreno"] = df["m2_construccion"] / df["m2_terreno"]

FEATURES_EXT = FEATURES_BASE + ["ratio_construccion_terreno"]

print(f"Mínimo m2_terreno: {df['m2_terreno'].min()} (sin ceros, sin riesgo de división)")
print(f"\nFeature nuevo — ratio_construccion_terreno:")
print(df["ratio_construccion_terreno"].describe().round(3))
print(f"\nFeatures extendido: {FEATURES_EXT}")
print(f"Total: {len(FEATURES_EXT)} (+ dummies de colonia en 3.3)")

Mínimo m2_terreno: 42 (sin ceros, sin riesgo de división)

Feature nuevo — ratio_construccion_terreno:
count    59.000
mean      0.948
std       0.342
min       0.383
25%       0.829
50%       0.949
75%       1.000
max       2.850
Name: ratio_construccion_terreno, dtype: float64

Features extendido: ['log_m2_construccion', 'recamaras', 'banos', 'estacionamientos', 'es_preventa', 'es_casa', 'tiene_piscina', 'tiene_cuarto_servicio', 'es_una_planta', 'tiene_mantenimiento_con_monto', 'ratio_construccion_terreno']
Total: 11 (+ dummies de colonia en 3.3)


### 3.3 Codificación de colonia y construcción de matrices X
Para RF y GB no se aplica `drop_first`: los modelos de árbol no sufren
la trampa de variable ficticia (no invierten una matriz) y pueden
aprovechar las 6 dummies completas. Se construyen cuatro matrices X:
base y extendida para cada modelo, todas con el mismo target `log_precio`.

In [5]:
dummies_colonia = pd.get_dummies(
    df[STRAT_COL], prefix="colonia", drop_first=False
).astype(int)

X_base = pd.concat([df[FEATURES_BASE], dummies_colonia], axis=1).astype(float)
X_ext  = pd.concat([df[FEATURES_EXT],  dummies_colonia], axis=1).astype(float)
y      = df[TARGET]

print("Columnas de dummies:", list(dummies_colonia.columns))
print(f"\nX_base : {X_base.shape}  — faltantes: {X_base.isna().sum().sum()}")
print(f"X_ext  : {X_ext.shape}  — faltantes: {X_ext.isna().sum().sum()}")
print(f"y      : {y.shape}")

Columnas de dummies: ['colonia_altabrisa', 'colonia_chuburna de hidalgo', 'colonia_francisco de montejo', 'colonia_garcia gineres', 'colonia_santa gertrudis copo', 'colonia_yucatan country club']

X_base : (59, 16)  — faltantes: 0
X_ext  : (59, 17)  — faltantes: 0
y      : (59,)


## 4. Etapa 1 — Modelos con feature set base
Se evalúan RF y GB con el mismo feature set del baseline OLS para aislar
el efecto del modelo. Se usa CV anidada: el loop externo
(RepeatedStratifiedKFold k=5, 10 repeticiones) estima el error de
generalización; el loop interno (GridSearchCV k=5) selecciona
hiperparámetros dentro de cada fold de entrenamiento. Esta separación
evita que el tuneo infle las métricas de evaluación.

### 4.1 Random Forest — feature set base

In [6]:
from sklearn.model_selection import GridSearchCV, KFold

param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth":    [None, 5],
    "min_samples_leaf": [1, 2],
}

cv_outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
cv_inner = KFold(n_splits=5, shuffle=True, random_state=42)  # ← KFold, no StratifiedKFold

rmse_rf_base = []
mae_rf_base  = []

for i, (train_idx, val_idx) in enumerate(cv_outer.split(X_base, df[STRAT_COL])):
    X_train, X_val = X_base.iloc[train_idx], X_base.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    gs = GridSearchCV(
        RandomForestRegressor(random_state=42),
        param_grid_rf,
        cv=cv_inner,
        scoring="neg_mean_squared_error",
        n_jobs=-1,
    )
    gs.fit(X_train, y_train)

    y_pred_orig = np.exp(gs.best_estimator_.predict(X_val))
    y_val_orig  = np.exp(y_val)

    rmse_rf_base.append(np.sqrt(mean_squared_error(y_val_orig, y_pred_orig)))
    mae_rf_base.append(mean_absolute_error(y_val_orig, y_pred_orig))

    if (i + 1) % 10 == 0:
        print(f"  Fold {i+1}/50 completado")

print(f"\nRF base — RMSE: {np.mean(rmse_rf_base)/1e6:.2f}M ± {np.std(rmse_rf_base)/1e6:.2f}M")
print(f"RF base — MAE:  {np.mean(mae_rf_base)/1e6:.2f}M ± {np.std(mae_rf_base)/1e6:.2f}M")

  Fold 10/50 completado
  Fold 20/50 completado
  Fold 30/50 completado
  Fold 40/50 completado
  Fold 50/50 completado

RF base — RMSE: 3.85M ± 2.44M
RF base — MAE:  2.11M ± 1.19M


### 4.2 Gradient Boosting — feature set base

In [7]:
param_grid_gb = {
    "n_estimators":  [100, 200],
    "max_depth":     [2, 3],
    "learning_rate": [0.05, 0.1],
}

rmse_gb_base = []
mae_gb_base  = []

for i, (train_idx, val_idx) in enumerate(cv_outer.split(X_base, df[STRAT_COL])):
    X_train, X_val = X_base.iloc[train_idx], X_base.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    gs = GridSearchCV(
        GradientBoostingRegressor(random_state=42),
        param_grid_gb,
        cv=cv_inner,
        scoring="neg_mean_squared_error",
        n_jobs=-1,
    )
    gs.fit(X_train, y_train)

    y_pred_orig = np.exp(gs.best_estimator_.predict(X_val))
    y_val_orig  = np.exp(y_val)

    rmse_gb_base.append(np.sqrt(mean_squared_error(y_val_orig, y_pred_orig)))
    mae_gb_base.append(mean_absolute_error(y_val_orig, y_pred_orig))

    if (i + 1) % 10 == 0:
        print(f"  Fold {i+1}/50 completado")

print(f"\nGB base — RMSE: {np.mean(rmse_gb_base)/1e6:.2f}M ± {np.std(rmse_gb_base)/1e6:.2f}M")
print(f"GB base — MAE:  {np.mean(mae_gb_base)/1e6:.2f}M ± {np.std(mae_gb_base)/1e6:.2f}M")

  Fold 10/50 completado
  Fold 20/50 completado
  Fold 30/50 completado
  Fold 40/50 completado
  Fold 50/50 completado

GB base — RMSE: 3.60M ± 2.23M
GB base — MAE:  2.02M ± 1.09M


## 5. Etapa 2 — Modelos con feature set extendido
Se agrega `ratio_construccion_terreno` al feature set base para medir su
aporte marginal. Si las métricas mejoran, el ratio captura información
útil que `log_m2_construccion` solo no recoge. Si no mejoran, el feature
no aporta valor adicional en este dataset.

### 5.1 Random Forest con ratio

In [8]:
rmse_rf_ext = []
mae_rf_ext  = []

for i, (train_idx, val_idx) in enumerate(cv_outer.split(X_ext, df[STRAT_COL])):
    X_train, X_val = X_ext.iloc[train_idx], X_ext.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    gs = GridSearchCV(
        RandomForestRegressor(random_state=42),
        param_grid_rf,
        cv=cv_inner,
        scoring="neg_mean_squared_error",
        n_jobs=-1,
    )
    gs.fit(X_train, y_train)

    y_pred_orig = np.exp(gs.best_estimator_.predict(X_val))
    y_val_orig  = np.exp(y_val)

    rmse_rf_ext.append(np.sqrt(mean_squared_error(y_val_orig, y_pred_orig)))
    mae_rf_ext.append(mean_absolute_error(y_val_orig, y_pred_orig))

    if (i + 1) % 10 == 0:
        print(f"  Fold {i+1}/50 completado")

print(f"\nRF ext — RMSE: {np.mean(rmse_rf_ext)/1e6:.2f}M ± {np.std(rmse_rf_ext)/1e6:.2f}M")
print(f"RF ext — MAE:  {np.mean(mae_rf_ext)/1e6:.2f}M ± {np.std(mae_rf_ext)/1e6:.2f}M")

  Fold 10/50 completado
  Fold 20/50 completado
  Fold 30/50 completado
  Fold 40/50 completado
  Fold 50/50 completado

RF ext — RMSE: 3.92M ± 2.48M
RF ext — MAE:  2.17M ± 1.21M


### 5.2 Gradient Boosting con ratio

In [9]:
rmse_gb_ext = []
mae_gb_ext  = []

for i, (train_idx, val_idx) in enumerate(cv_outer.split(X_ext, df[STRAT_COL])):
    X_train, X_val = X_ext.iloc[train_idx], X_ext.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    gs = GridSearchCV(
        GradientBoostingRegressor(random_state=42),
        param_grid_gb,
        cv=cv_inner,
        scoring="neg_mean_squared_error",
        n_jobs=-1,
    )
    gs.fit(X_train, y_train)

    y_pred_orig = np.exp(gs.best_estimator_.predict(X_val))
    y_val_orig  = np.exp(y_val)

    rmse_gb_ext.append(np.sqrt(mean_squared_error(y_val_orig, y_pred_orig)))
    mae_gb_ext.append(mean_absolute_error(y_val_orig, y_pred_orig))

    if (i + 1) % 10 == 0:
        print(f"  Fold {i+1}/50 completado")

print(f"\nGB ext — RMSE: {np.mean(rmse_gb_ext)/1e6:.2f}M ± {np.std(rmse_gb_ext)/1e6:.2f}M")
print(f"GB ext — MAE:  {np.mean(mae_gb_ext)/1e6:.2f}M ± {np.std(mae_gb_ext)/1e6:.2f}M")

  Fold 10/50 completado
  Fold 20/50 completado
  Fold 30/50 completado
  Fold 40/50 completado
  Fold 50/50 completado

GB ext — RMSE: 3.81M ± 2.34M
GB ext — MAE:  2.14M ± 1.15M


## 6. Tabla comparativa

In [10]:
resultados = pd.DataFrame({
    "Modelo": [
        "OLS baseline",
        "Random Forest (base)",
        "Gradient Boosting (base)",
        "Random Forest (+ ratio)",
        "Gradient Boosting (+ ratio)",
    ],
    "Feature set": [
        "base", "base", "base", "extendido", "extendido"
    ],
    "RMSE media (M)": [
        6.90,
        np.mean(rmse_rf_base)/1e6,
        np.mean(rmse_gb_base)/1e6,
        np.mean(rmse_rf_ext)/1e6,
        np.mean(rmse_gb_ext)/1e6,
    ],
    "RMSE std (M)": [
        2.05,
        np.std(rmse_rf_base)/1e6,
        np.std(rmse_gb_base)/1e6,
        np.std(rmse_rf_ext)/1e6,
        np.std(rmse_gb_ext)/1e6,
    ],
    "MAE media (M)": [
        3.19,
        np.mean(mae_rf_base)/1e6,
        np.mean(mae_gb_base)/1e6,
        np.mean(mae_rf_ext)/1e6,
        np.mean(mae_gb_ext)/1e6,
    ],
    "MAE std (M)": [
        0.84,
        np.std(mae_rf_base)/1e6,
        np.std(mae_gb_base)/1e6,
        np.std(mae_rf_ext)/1e6,
        np.std(mae_gb_ext)/1e6,
    ],
}).round(2)

print(resultados.to_string(index=False))
print(f"\nModelo seleccionado: Gradient Boosting (feature set base)")
print(f"RMSE: {np.mean(rmse_gb_base)/1e6:.2f}M  |  MAE: {np.mean(mae_gb_base)/1e6:.2f}M")

                     Modelo Feature set  RMSE media (M)  RMSE std (M)  MAE media (M)  MAE std (M)
               OLS baseline        base            6.90          2.05           3.19         0.84
       Random Forest (base)        base            3.85          2.44           2.11         1.19
   Gradient Boosting (base)        base            3.60          2.23           2.02         1.09
    Random Forest (+ ratio)   extendido            3.92          2.48           2.17         1.21
Gradient Boosting (+ ratio)   extendido            3.81          2.34           2.14         1.15

Modelo seleccionado: Gradient Boosting (feature set base)
RMSE: 3.60M  |  MAE: 2.02M


## 7. Conclusiones

### Resultado principal
Gradient Boosting con feature set base es el modelo seleccionado:
RMSE=3.60M ± 2.23M MXN y MAE=2.02M ± 1.09M MXN en validación cruzada
repetida (k=5, 10 repeticiones). Representa una mejora del 48% en RMSE
y 37% en MAE sobre el baseline lineal.

### Hallazgos por etapa

**Etapa 1 — efecto del modelo:** ambos modelos de árbol superan
sustancialmente al baseline OLS. GB aventaja a RF por un margen pequeño
pero consistente en ambas métricas (RMSE 3.60M vs 3.85M, MAE 2.02M vs
2.11M). La ventaja de GB sobre RF es esperable: la construcción secuencial
de árboles corrigiendo errores previos es más efectiva que el ensamble
paralelo de RF en datasets pequeños.

**Etapa 2 — efecto del feature:** `ratio_construccion_terreno` no mejoró
ningún modelo. Con n=59, la información de densidad de edificación ya
queda capturada implícitamente por `log_m2_construccion` y las dummies
de colonia. Agregar el ratio introduce ruido marginal sin señal adicional.

### Nota sobre comparabilidad del protocolo
El baseline OLS se evaluó con `StratifiedKFold(k=5)` simple; los modelos
de árbol usan `RepeatedStratifiedKFold(k=5, n_repeats=10)`. Las
estimaciones del error no son directamente comparables en varianza —
el protocolo repetido captura más variabilidad entre particiones. La
dirección de la mejora es clara e inequívoca; la magnitud exacta de
la diferencia debe interpretarse con esta limitación en mente.

### Limitaciones
- n=59 limita la potencia estadística: las diferencias entre RF y GB
  son pequeñas relativas a la desviación estándar entre folds.
- El error absoluto sigue siendo alto (MAE 2.02M sobre una mediana de
  ~4.8M): el modelo es una prueba de concepto metodológica, no un
  servicio de valuación desplegable.
- Los hiperparámetros se exploraron en una grilla pequeña por
  restricción de tiempo de cómputo con n_repeats=10.

### Próximos pasos sugeridos
Recolectar más propiedades (n≥200) para reducir la varianza de las
estimaciones y explorar grillas de hiperparámetros más amplias.
La infraestructura metodológica (CV anidada, protocolo sin fuga,
métricas en escala original) está lista para escalar.

## Exportación del modelo final — Fase 5

In [3]:
# Exportación del modelo para la API— Fase 5
# Entrena GB sobre el dataset completo (59 obs) y serializa.

import joblib

# 1. Cargar dataset de modelado
df = pd.read_csv("../data/propiedades_modelo.csv")

# 2. Preparar features (idéntico al notebook comparativo)
df["log_m2_construccion"] = np.log(df["m2_construccion"])

dummies_colonia = pd.get_dummies(df["colonia"], prefix="colonia", dtype=int)

features_base = [
    "log_m2_construccion", "recamaras", "banos", "estacionamientos",
    "es_preventa", "es_casa", "tiene_piscina", "tiene_cuarto_servicio",
    "es_una_planta", "tiene_mantenimiento_con_monto",
]

X = pd.concat([df[features_base], dummies_colonia], axis=1)
y = df["log_precio"]

# 3. Entrenar sobre dataset completo
gb_final = GradientBoostingRegressor(random_state=42)
gb_final.fit(X, y)

# 4. Serializar modelo y lista de columnas
joblib.dump(gb_final, "../modelo_gb.pkl")
joblib.dump(X.columns.tolist(), "../columnas_modelo.pkl")

print(f"✓ Modelo guardado: modelo_gb.pkl")
print(f"✓ Columnas guardadas: columnas_modelo.pkl ({X.shape[1]} features)")
print(f"  {X.columns.tolist()}")

✓ Modelo guardado: modelo_gb.pkl
✓ Columnas guardadas: columnas_modelo.pkl (16 features)
  ['log_m2_construccion', 'recamaras', 'banos', 'estacionamientos', 'es_preventa', 'es_casa', 'tiene_piscina', 'tiene_cuarto_servicio', 'es_una_planta', 'tiene_mantenimiento_con_monto', 'colonia_altabrisa', 'colonia_chuburna de hidalgo', 'colonia_francisco de montejo', 'colonia_garcia gineres', 'colonia_santa gertrudis copo', 'colonia_yucatan country club']
